# Cuaderno histórico de 2022
Material de referencia, sin ejecución validada con bibliotecas actuales. Las salidas se han borrado para no volver a difundir identificadores de envíos.
Para la versión reproducible usa `00_analisis_reproducible.ipynb` o `python analysis.py Peski.xlsx`. Los ejemplos actuales son sintéticos.


In [ ]:
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
df_160 = pd.read_excel('datos predict.xlsx')
df_160

In [ ]:
import matplotlib.pyplot as plt
y1 = df_160['unidad']
x1 = df_160['date']
fig, ax = plt.subplots(figsize=(20,10))
plt.grid()
ax.plot(x1, y1, linewidth=2.0, label="total")
ax.xaxis.set_tick_params(rotation=40)
ax.set_ylabel('Unidades',fontsize = 20)
ax.set_title('Cuatrimestre',fontsize = 40)
ax.legend(fontsize = 20)

In [ ]:
flight_data=df_160
flight_data.columns


In [ ]:
all_data = flight_data['unidad'].values.astype(float)


In [ ]:
print(all_data)

In [ ]:
test_data_size = 5

train_data = all_data[:-test_data_size]
test_data = all_data[-test_data_size:]


In [ ]:
print(len(train_data))
print(len(test_data))

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(-1, 1))
train_data_normalized = scaler.fit_transform(train_data .reshape(-1, 1))


In [ ]:
print(train_data_normalized[:5])
print(train_data_normalized[-5:])


In [ ]:
train_data_normalized = torch.FloatTensor(train_data_normalized).view(-1)
train_data_normalized

In [ ]:
train_window = 20


In [ ]:
def create_inout_sequences(input_data, tw):
    inout_seq = []
    L = len(input_data)
    for i in range(L-tw):
        train_seq = input_data[i:i+tw]
        train_label = input_data[i+tw:i+tw+1]
        inout_seq.append((train_seq ,train_label))
    return inout_seq


In [ ]:
train_inout_seq = create_inout_sequences(train_data_normalized, train_window)


In [ ]:
train_inout_seq[:5]


In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size=1, hidden_layer_size=100, output_size=1):
        super().__init__()
        self.hidden_layer_size = hidden_layer_size

        self.lstm = nn.LSTM(input_size, hidden_layer_size)

        self.linear = nn.Linear(hidden_layer_size, output_size)

        self.hidden_cell = (torch.zeros(1,1,self.hidden_layer_size),
                            torch.zeros(1,1,self.hidden_layer_size)) # (num_layers * num_directions, batch_size, hidden_size)

    def forward(self, input_seq):
        lstm_out, self.hidden_cell = self.lstm(input_seq.view(len(input_seq) ,1, -1), self.hidden_cell)
        predictions = self.linear(lstm_out.view(len(input_seq), -1))
        return predictions[-1]


In [ ]:
model = LSTM()
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
print(model)


In [ ]:
epochs = 150

for i in range(epochs):
    for seq, labels in train_inout_seq:
        optimizer.zero_grad()
        model.hidden_cell = (torch.zeros(1, 1, model.hidden_layer_size),
                        torch.zeros(1, 1, model.hidden_layer_size))

        y_pred = model(seq)

        single_loss = loss_function(y_pred, labels)
        single_loss.backward()
        optimizer.step()

    if i%25 == 1:
        print(f'epoch: {i:3} loss: {single_loss.item():10.8f}')

print(f'epoch: {i:3} loss: {single_loss.item():10.10f}')


In [ ]:
fut_pred = 20

test_inputs = train_data_normalized[-train_window:].tolist()
print(test_inputs)


In [ ]:
model.eval()

for i in range(fut_pred):
    seq = torch.FloatTensor(test_inputs[-train_window:])
    with torch.no_grad():
        model.hidden_cell = (torch.zeros(1, 1, model.hidden_layer_size),
                        torch.zeros(1, 1, model.hidden_layer_size))
        test_inputs.append(model(seq).item())


In [ ]:
test_inputs[fut_pred:]


In [ ]:
actual_predictions = scaler.inverse_transform(np.array(test_inputs[train_window:] ).reshape(-1, 1))
print(actual_predictions)


In [ ]:
x = np.arange(118, 138, 1)
print(x)


In [ ]:
fig, ax = plt.subplots(figsize=(20,10))


plt.xlabel('Serie temporal',fontsize = 20)
ax.set_ylabel('Unidades',fontsize = 20)
ax.set_title('PREDICCION',fontsize = 40)
plt.grid(True)
plt.autoscale(axis='x', tight=True)
plt.plot(df_160['unidad'], linewidth=2.0 ,label='REAL')
plt.plot(x,actual_predictions, linewidth=4.0 ,label='PREDICT')
ax.legend(fontsize = 20)
plt.savefig("static\Predict cuatrimestre.jpg")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(20,10))
plt.xlabel('Serie temporal',fontsize = 20)
ax.set_ylabel('Unidades',fontsize = 20)
ax.set_title('Modelo Window',fontsize = 40)
plt.grid(True)
plt.autoscale(axis='x', tight=True)
plt.plot(df_160['unidad'][-train_window:], linewidth=5.0 ,label='REAL')
plt.plot(x,actual_predictions, linewidth=5.0 ,label='PREDICT')
ax.legend(fontsize = 20)
#plt.savefig("static\Modelo window.jpg")


In [ ]:
print(x)

In [ ]:
x=[27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46]

In [ ]:
print(actual_predictions)

In [ ]:
import pandas as pd
predict = pd.DataFrame(actual_predictions)
predict

In [ ]:
import pandas as pd


rt = pd.DataFrame({"Mes": (10,10,10,10,10,11,11,11,11,11,11,11,11,11,11,11,11,11,11,11),"Dia": (27,28,29,30,31,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15)})
year = __import__("datetime").datetime.now().year
rt['Date']=pd.to_datetime({'year': [year] * rt.shape[0],'month': rt["Mes"],'day': rt['Dia']})
rt

In [ ]:
para_predict = pd.concat([rt,predict], axis=1)
para_predict.columns = ['mes','dia','date','unidad']
para_predict

In [ ]:
para_predict.to_excel("para_predict.xlsx")